# Tutorial 06 — DICOM RT-STRUCT Export

**Audience:** Radiation oncologists, medical physicists, and PhD students  
**GPU required:** No — works on any pre-generated output in `outputs/`  
**Time:** ~10 minutes  

---

## What this tutorial covers

The NVIDIA MAISI model generates synthetic CT volumes and segmentation masks as NIfTI files (`.nii.gz`). That format is great for research but **cannot be loaded into a treatment planning system (TPS)**.

This tutorial shows how MAISI-RT converts those NIfTI files into two clinical DICOM objects:

| DICOM object | Modality tag | What it contains |
|---|---|---|
| CT image series | `CT` | One `.dcm` file per axial slice — identical to scanner output |
| RT-STRUCT | `RTSTRUCT` | Contour sequences, one ROI per segmented structure |

The result can be opened in **3D Slicer** (free) or imported into **Eclipse, RayStation, or Monaco** via standard DICOM import.

---

## Clinical background — what is an RT-STRUCT?

In radiotherapy, a **structure set** (RT-STRUCT) is the DICOM file that stores all the delineated volumes:

- **GTV** — Gross Tumour Volume (visible disease on imaging)
- **CTV** — Clinical Target Volume (GTV + microscopic spread margin)
- **PTV** — Planning Target Volume (CTV + setup uncertainty margin)
- **OARs** — Organs at Risk (structures to protect: spinal cord, bladder, rectum, etc.)

An RT-STRUCT file stores the 3D contours as slice-by-slice polygon sequences, each linked to the reference CT via DICOM UIDs. This is how the TPS knows which pixels to use when calculating dose constraints.

**What we're doing here** is generating synthetic anatomy with MAISI and packaging it in exactly this clinical format — useful for:
- Demonstrating auto-segmentation pipelines without patient data
- Training staff to use a TPS with realistic (but synthetic) cases
- Building ground-truth datasets for auto-contouring model development
- Teaching medical physics students about structure-set geometry

## Step 1 — Check what's available to export

In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, "../scripts/export_rt_struct.py", "--list"],
    capture_output=True, text=True, cwd=".."
)
print(result.stdout or result.stderr)

## Step 2 — Export a pelvis RT case

The `ct_pelvis_rt` preset generates prostate, bladder, femoral heads, hip bones, sacrum, colon, and small bowel — the standard OAR set for prostate IMRT/VMAT planning.

The exporter does two things:
1. Converts the NIfTI CT → 128 DICOM CT slices (one per axial plane)
2. Converts the integer label mask → DICOM RT-STRUCT with named, colour-coded ROIs

In [ ]:
result = subprocess.run(
    [sys.executable, "../scripts/export_rt_struct.py", "--preset", "ct_pelvis_rt"],
    capture_output=True, text=True, cwd=".."
)
print(result.stdout or result.stderr)

## Step 3 — Inspect the output directory

In [ ]:
from pathlib import Path

export_root = sorted(Path("../outputs/dicom_export").glob("ct_seed7*"))[-1]
print("Export directory:", export_root)
print()

ct_files  = sorted((export_root / "CT").glob("*.dcm"))
rt_files  = sorted((export_root / "RT").glob("*.dcm"))

print(f"  CT/   — {len(ct_files)} DICOM slices  ({ct_files[0].stat().st_size // 1024} KB each)")
print(f"  RT/   — {len(rt_files)} RT-STRUCT file ({rt_files[0].stat().st_size // 1024} KB)")
print(f"  manifest.json — structure summary")

## Step 4 — Inspect the DICOM CT metadata

In [ ]:
import pydicom

ct_slice = pydicom.dcmread(str(ct_files[64]))  # mid-volume slice

print("=== DICOM CT metadata (slice 65/128) ===")
print(f"Modality              : {ct_slice.Modality}")
print(f"Patient name          : {ct_slice.PatientName}")
print(f"Study description     : {ct_slice.StudyDescription}")
print(f"Rows × Columns        : {ct_slice.Rows} × {ct_slice.Columns}")
print(f"Pixel spacing (mm)    : {[float(v) for v in ct_slice.PixelSpacing]}")
print(f"Slice thickness (mm)  : {float(ct_slice.SliceThickness):.3f}")
print(f"Image pos. patient    : {[float(v) for v in ct_slice.ImagePositionPatient]}")
print(f"Image orientation     : {[float(v) for v in ct_slice.ImageOrientationPatient]}")
print(f"Rescale intercept     : {ct_slice.RescaleIntercept}  (HU = stored_value + intercept)")
print(f"Rescale slope         : {ct_slice.RescaleSlope}")
print(f"Bits allocated        : {ct_slice.BitsAllocated}")

### Why these tags matter in radiotherapy

| Tag | RT significance |
|---|---|
| `PixelSpacing` | Used by TPS to scale DVH volumes; inaccurate spacing → wrong dose-volume statistics |
| `ImagePositionPatient` | Defines the 3D coordinate frame shared between CT and RT-STRUCT |
| `ImageOrientationPatient` | Must agree across all CT slices; used to map contour polygons to patient space |
| `RescaleIntercept` | Converts stored pixels → HU; HU values feed the electron density table for dose calc |
| `BitsAllocated` | 16-bit signed int allows the full HU range (−1024 to ≈3000) without clipping |

## Step 5 — Inspect the RT-STRUCT

In [ ]:
rs = pydicom.dcmread(str(rt_files[0]))

print(f"Modality     : {rs.Modality}")
print(f"Series desc. : {rs.SeriesDescription}")
print(f"Number of ROIs : {len(rs.StructureSetROISequence)}")
print()
print("ROI list:")
for roi in rs.StructureSetROISequence:
    print(f"  ROI {roi.ROINumber:2d}  {roi.ROIName}")

In [ ]:
# Each ROI has a colour and contour sequences (slice-by-slice polygons)
print("Colour and contour summary:")
print()
print(f"{'ROI':<30} {'Color (R,G,B)':>15} {'Contour slices':>16}")
print("-" * 65)

for roi_obs in rs.RTROIObservationsSequence:
    roi_num = roi_obs.ReferencedROINumber
    # Find matching name
    name = next(r.ROIName for r in rs.StructureSetROISequence if r.ROINumber == roi_num)

    # Find colour
    contour_entry = next(
        (c for c in rs.ROIContourSequence if c.ReferencedROINumber == roi_num), None
    )
    color = contour_entry.ROIDisplayColor if contour_entry else "—"
    n_slices = len(contour_entry.ContourSequence) if contour_entry and hasattr(contour_entry, 'ContourSequence') else 0

    print(f"{name:<30} {str(list(color)):>15} {n_slices:>16} slices")

## Step 6 — Read the manifest (volumes + QUANTEC notes)

In [ ]:
import json

manifest = json.loads((export_root / "manifest.json").read_text())

print(f"{'Structure':<30} {'Volume (mL)':>12}   QUANTEC / dose constraint note")
print("-" * 90)
for s in manifest["structures"]:
    if s["exported"]:
        note = s["quantec_note"]
        if note == "—":
            note = ""
        print(f"{s['name']:<30} {s['volume_ml']:>12.1f}   {note}")

### What do these volume numbers tell us?

These are the synthetic organ volumes computed from the MAISI-generated label mask. For reference:

| Structure | Typical clinical range | Our synthetic value |
|---|---|---|
| Prostate | 20–80 mL (healthy adult) | see above |
| Bladder (planning, partly filled) | 100–500 mL | see above |
| Rectum | 50–200 mL | — |

> The model does not enforce realistic size distributions. You may see unusually large or small structures — this is a known limitation. Running multiple seeds (Tutorial 05) gives you a distribution to assess variability.

## Step 7 — Visualise a contour overlay

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import nibabel as nib

# Load data straight from NIfTI (faster than reading 128 DICOMs for a plot)
ct_nii    = nib.load(sorted(Path("../outputs/maisi2_showcase_ct_pelvis_rt").rglob("*_image.nii.gz"))[0])
label_nii = nib.load(sorted(Path("../outputs/maisi2_showcase_ct_pelvis_rt").rglob("*_label.nii.gz"))[0])

ct_data    = ct_nii.get_fdata(dtype=np.float32)
label_data = label_nii.get_fdata(dtype=np.float32)

# Anatomy list for this preset (1-indexed)
anatomy_list = [
    "prostate", "bladder", "left femur", "right femur",
    "left hip", "right hip", "sacrum", "colon", "small bowel"
]

# Clinical colours (matching the RT-STRUCT export)
COLORS = {
    "prostate":    [0, 80, 255],
    "bladder":     [173, 216, 230],
    "left femur":  [147, 112, 219],
    "right femur": [130, 95, 210],
    "left hip":    [160, 120, 230],
    "right hip":   [150, 110, 220],
    "sacrum":      [240, 215, 170],
    "colon":       [200, 100, 50],
    "small bowel": [180, 120, 60],
}

# Find the axial slice with the most structure coverage
structure_presence = (label_data > 0).sum(axis=(0, 1))  # per-slice count
best_slice = int(np.argmax(structure_presence))

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
slices = [best_slice - 15, best_slice, best_slice + 15]
slices = [max(0, min(ct_data.shape[2]-1, s)) for s in slices]

for ax, k in zip(axes, slices):
    # CT background
    ct_slice = ct_data[:, :, k].T
    ax.imshow(ct_slice, cmap="gray", vmin=-200, vmax=400, origin="upper")

    # Structure overlays
    overlay = np.zeros((*ct_slice.shape, 4), dtype=float)  # RGBA
    for lab_idx, name in enumerate(anatomy_list, start=1):
        mask = (label_data[:, :, k] == lab_idx).T
        if not mask.any():
            continue
        c = COLORS.get(name, [200, 200, 200])
        overlay[mask, 0] = c[0] / 255
        overlay[mask, 1] = c[1] / 255
        overlay[mask, 2] = c[2] / 255
        overlay[mask, 3] = 0.45  # semi-transparent

    ax.imshow(overlay, origin="upper")
    ax.set_title(f"Axial slice {k + 1}/128", fontsize=11)
    ax.axis("off")

# Legend
patches = [
    mpatches.Patch(color=[v/255 for v in rgb], label=name)
    for name, rgb in COLORS.items()
]
fig.legend(handles=patches, loc="lower center", ncol=5, fontsize=9,
           bbox_to_anchor=(0.5, -0.05), framealpha=0.9)

fig.suptitle("Synthetic Pelvis CT — Structure Overlays (matching RT-STRUCT colours)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("../figures/fig5_rt_struct_overlay.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → figures/fig5_rt_struct_overlay.png")

## Step 8 — Load in 3D Slicer

3D Slicer is a free, open-source medical imaging platform used across research and clinical training:

1. Download from [slicer.org](https://www.slicer.org) if not already installed  
2. Open Slicer → **File → Add DICOM Data**
3. Click **Import** and select the `CT/` directory  
   - Slicer will import all 128 slices as a CT volume  
4. Then **File → Add DICOM Data** again → import the `RT/RS_*.dcm` file  
   - Choose **Segmentation** when prompted for the type  
5. In the **Segment Editor**, toggle structure visibility with the eye icon  

You will see the synthetic CT with all structures colour-coded exactly as in the RT-STRUCT.

In [ ]:
# Print paths for copy-pasting into Slicer
print("Paths to load in 3D Slicer:")
print()
print("CT series directory:")
print(" ", export_root / "CT")
print()
print("RT-STRUCT file:")
print(" ", rt_files[0])

## Step 9 — Export all four CT presets at once

In [ ]:
presets = ["ct_chest_cardio_lung", "ct_abdomen_organs_tumor", "ct_head_neck", "ct_pelvis_rt"]

for preset in presets:
    print(f"Exporting {preset} ...", end=" ", flush=True)
    r = subprocess.run(
        [sys.executable, "../scripts/export_rt_struct.py", "--preset", preset, "--quiet"],
        capture_output=True, text=True, cwd=".."
    )
    if r.returncode == 0:
        print("done")
    else:
        print("FAILED")
        print(r.stderr[:300])

In [ ]:
# Summary of all exported structure sets
import json
from pathlib import Path

print(f"{'Directory':<35} {'Structures':>11} {'Total volume (mL)':>18}")
print("-" * 70)

for mpath in sorted(Path("../outputs/dicom_export").rglob("manifest.json")):
    data = json.loads(mpath.read_text())
    structs = [s for s in data["structures"] if s["exported"]]
    total_vol = sum(s["volume_ml"] for s in structs)
    print(f"{mpath.parent.name:<35} {len(structs):>11} {total_vol:>18.1f}")

## What this means for your research

You now have a complete pipeline from **synthetic image generation → clinical DICOM format**:

```
python MAISI_RT_Generate.py --preset ct_pelvis_rt
          ↓
python scripts/export_rt_struct.py --preset ct_pelvis_rt
          ↓
outputs/dicom_export/
  CT/        ← import to TPS or Slicer
  RT/RS_*.dcm ← RT-STRUCT with named OARs + QUANTEC notes
  manifest.json
```

**Use cases for each audience:**

| Audience | How to use this export |
|---|---|
| Radiation oncologist | Load in Eclipse/RayStation for contouring practice or TPS demonstration without patient data |
| Medical physicist | Test auto-planning constraints, evaluate synthetic organ volumes against published literature |
| PhD student | Build labelled CT + RT-STRUCT datasets for auto-segmentation model training/evaluation |

---

**Next:** Tutorial 07 (coming) — Structure geometry metrics: computing DSC, Hausdorff distance, and volume statistics across a multi-seed synthetic cohort.